# Reto Datos: Pipeline ETL con PySpark para BCS en vacas lecheras


El objetivo es extraer, limpiar, transformar y cargar imágenes de vacas con su condición corporal (BCS), y dividirlas en folds para validación cruzada agrupando por vaca. Todo con PySpark, tanto en local como en el clúster del equipo.

Respecto a los datos como las fotos del CAETEC son confidenciales, para la demostración usamos una muestra del dataset público Dairy cow body condition score target detection data set (ScienceDB, DOI [10.57760/sciencedb.16704](https://doi.org/10.57760/sciencedb.16704)). El pipeline funciona igual con los datos del rancho.

En Colab, la primera celda de código monta Drive e instala lo que falta. En local:
```bash
pip install pyspark==3.5.* pillow pillow-heif scikit-learn pyarrow
```
Clúster: Spark 3.5.0 con Java 11 en spark://spark-master:7077. No hay HDFS ni almacenamiento compartido, así que la muestra tiene que estar en la misma ruta en las 3 máquinas por el momento.

### Estructura de los datos
```
DATA_PATH/
├── 3.25/   GS_<vaca>_<n>.jpg   (y L-i####.jpg / R-i####.jpg)
└── 3.5/    GS_<vaca>_<n>.jpg   ...
```
El BCS sale del nombre de la carpeta y el ID de la vaca sale del nombre del archivo (GS_vaca_n).


## Paso 0: Configuración del entorno

Solo para Colab: montamos Drive (pide permiso la primera vez) e instalamos lo que falta. En local o en el clúster esta celda no hace nada.

In [2]:
import subprocess, sys
try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.*", "pillow-heif"], check=True)
print("Colab con Drive montado" if EN_COLAB else "No es Colab: las dependencias se instalan a mano")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab con Drive montado


In [3]:
# ===================== CONFIGURACIÓN =====================
MODO = "local"            # "local" (Colab / laptop)  |  "cluster" (spark://spark-master:7077)

# Ruta de la muestra de ~100 fotos que procesa el pipeline (se arma sola en la celda siguiente).
# En modo clúster tiene que estar en la misma ruta en las 3 máquinas.
DATA_PATH = "/content/drive/MyDrive/lecheros/Dataset/muestra_100"     # Colab (Drive montado)
# DATA_PATH = "/tmp/bcs_sample"                                        # clúster

# Carpeta de resultados: clean/ (imágenes), processed/ (tablas) y sample/ (muestra para el repo).
# En Colab va en Drive para que no se borre al cerrar la sesión. Se crea sola, no hay que hacerla a mano.
OUT_PATH = "/content/drive/MyDrive/lecheros/salidas_etl"            # Colab
# OUT_PATH = "salidas"                                                # clúster (se escribe en el driver)
CLASES_BCS = [3.25, 3.5]  # clases de ScienceDB usadas en la demostración
VACAS_POR_CLASE = None    # None = todas las vacas de la carpeta; un número = muestra de N vacas por clase
K = 5                     # folds
SEED = 42
LADO_MAX = 640            # px del lado mayor de las imágenes procesadas
N_MUESTRA_REPO = 20       # imágenes que se copian a data/sample/ para el repositorio

### Paso previo: armar la muestra de ~100 fotos
El dataset chino completo tiene unas 20 mil fotos. Para la simulación copiamos ~100 a `DATA_PATH`, tomando vacas completas (todas sus fotos), la misma cantidad por clase y al menos `K` vacas por clase para que haya una en cada fold. Se eligen al azar con `SEED`, así que siempre salen las mismas.

Además copiamos una toma `L-`/`R-` por clase para que el Paso 3 tenga algo que excluir, y dejamos fuera los números de vaca que aparecen en las dos carpetas. Si la muestra ya existe, esta celda no copia nada.

In [4]:
import os, re, random, shutil
from pathlib import Path

ORIGEN = Path("/content/drive/MyDrive/lecheros/Dataset/Dataset chino")   # dataset completo (~20 mil fotos)
FOTOS_POR_CLASE = 50                                                       # ~100 fotos en total
destino = Path(DATA_PATH)

if destino.exists() and any(destino.rglob("*.jpg")):
    print(f"La muestra ya existe en {destino}: no se vuelve a copiar")
elif not ORIGEN.exists():
    print(f"No encontré {ORIGEN}: revisen la ruta del dataset completo")
else:
    gs = re.compile(r"^GS_(\d+)_\d+\.jpe?g$", re.IGNORECASE)
    archivos, vacas = {}, {}
    for c in CLASES_BCS:
        archivos[c] = sorted(os.listdir(ORIGEN / str(c)))
        vacas[c] = {}
        for nombre in archivos[c]:
            m = gs.match(nombre)
            if m:
                vacas[c].setdefault(m.group(1), []).append(nombre)
    repetidas = set.intersection(*(set(v) for v in vacas.values()))
    print("Vacas por carpeta:", {c: len(v) for c, v in vacas.items()},
          "| números de vaca repetidos entre carpetas:", len(repetidas))

    rng = random.Random(SEED)
    total = 0
    for c in CLASES_BCS:
        candidatas = sorted(set(vacas[c]) - repetidas)
        rng.shuffle(candidatas)
        elegidas, n = [], 0
        for v in candidatas:
            if n >= FOTOS_POR_CLASE and len(elegidas) >= K:
                break
            elegidas.append(v)
            n += len(vacas[c][v])
        (destino / str(c)).mkdir(parents=True, exist_ok=True)
        for v in elegidas:
            for nombre in vacas[c][v]:
                shutil.copy2(ORIGEN / str(c) / nombre, destino / str(c) / nombre)
        # una toma L-/R- (sin ID de vaca) para que la limpieza tenga algo que excluir
        oblicua = next((x for x in archivos[c] if re.match(r"^[LR]-", x)), None)
        if oblicua:
            shutil.copy2(ORIGEN / str(c) / oblicua, destino / str(c) / oblicua)
            n += 1
        print(f"{c}: {len(elegidas)} vacas, {n} fotos copiadas")
        total += n
    print(f"Muestra creada en {destino}: {total} fotos")

Vacas por carpeta: {3.25: 519, 3.5: 804} | números de vaca repetidos entre carpetas: 8
3.25: 8 vacas, 60 fotos copiadas
3.5: 6 vacas, 54 fotos copiadas
Muestra creada en /content/drive/MyDrive/lecheros/Dataset/muestra_100: 114 fotos


Solo para modo clúster: hay que copiar la muestra a la misma ruta en las 3 máquinas y también en la del driver, porque los Pasos 6 a 8 leen los archivos desde ahí. Es lo mismo que hicimos con el CSV en el ejercicio de churn:
```bash
cp -r bcs_sample /tmp/bcs_sample                       # driver (esta Mac)
for h in spark-master spark-worker1 spark-worker2; do rsync -a bcs_sample/ $h:/tmp/bcs_sample/; done
```
Nota: binaryFile ignora los archivos de 0 bytes, así que esos no salen en el conteo.

In [5]:
import os, sys, glob, socket, hashlib, json, shutil
from pathlib import Path
import pandas as pd

if MODO == "cluster":
    # Como en el notebook de churn: el driver usa Spark 3.5.0 y Java 11, igual que el clúster
    os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"
    os.environ["SPARK_HOME"] = "/opt/spark"
    sys.path[:0] = ["/opt/spark/python", glob.glob("/opt/spark/python/lib/py4j-*-src.zip")[0]]
    MASTER_URL = "spark://spark-master:7077"
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.connect(("spark-master", 7077))
        DRIVER_IP = s.getsockname()[0]
else:
    MASTER_URL, DRIVER_IP = "local[*]", None
    # En Colab, Drive ya quedó montado en la primera celda

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

builder = SparkSession.builder.master(MASTER_URL).appName("BCS_ETL").config("spark.sql.shuffle.partitions", "8")
if DRIVER_IP:
    builder = builder.config("spark.driver.host", DRIVER_IP)
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(f"PySpark {spark.version} | Master: {spark.sparkContext.master}")

PySpark 3.5.9 | Master: local[*]


## Paso 1: Extracción

Con binaryFile cada imagen se lee como una fila con su ruta, fecha de modificación, tamaño y contenido en binario. Es como spark.read.csv, pero para datos no estructurados.

In [6]:
raw = (spark.read.format("binaryFile")
       .option("recursiveFileLookup", "true")
       .option("pathGlobFilter", "*.{jpg,jpeg,JPG,JPEG,png,PNG,heic,HEIC}")
       .load(DATA_PATH))

raw.select("path", "modificationTime", "length").show(5, truncate=60)
raw.printSchema()
n_raw = raw.count()
print(f"Archivos leídos: {n_raw}")

+------------------------------------------------------------+-------------------+------+
|                                                        path|   modificationTime|length|
+------------------------------------------------------------+-------------------+------+
|file:/content/drive/MyDrive/lecheros/Dataset/muestra_100/...|2024-01-24 02:48:36| 88759|
|file:/content/drive/MyDrive/lecheros/Dataset/muestra_100/...|2024-01-24 02:48:41| 75140|
|file:/content/drive/MyDrive/lecheros/Dataset/muestra_100/...|2024-01-23 10:10:24| 69114|
|file:/content/drive/MyDrive/lecheros/Dataset/muestra_100/...|2024-01-23 10:10:24| 68862|
|file:/content/drive/MyDrive/lecheros/Dataset/muestra_100/...|2024-01-23 10:10:24| 67577|
+------------------------------------------------------------+-------------------+------+
only showing top 5 rows

root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true

## Paso 2: Parseo de metadatos

Con expresiones regulares de Spark sacamos:
- bcs: la carpeta donde está la imagen (por ejemplo .../3.25/...).
- cow_id: el número de vaca del nombre GS_vaca_n. Las tomas L- y R- no tienen ID de vaca, así que no se pueden agrupar y las dejamos fuera.
- image_id: el hash SHA-256 del contenido. Si dos archivos tienen el mismo hash, son la misma imagen.

In [7]:
df = (raw
      .withColumn("filename", F.regexp_extract("path", r"([^/]+)$", 1))
      .withColumn("bcs", F.regexp_extract("path", r"/(\d+(?:\.\d+)?)/[^/]+$", 1).cast("double"))
      .withColumn("cow_id", F.regexp_extract("filename", r"^GS_(\d+)_\d+", 1))
      .withColumn("cow_id", F.when(F.col("cow_id") == "", None).otherwise(F.col("cow_id")))
      .withColumn("tipo_toma", F.when(F.col("filename").rlike(r"^GS_"), "cenital_trasera")
                                .when(F.col("filename").rlike(r"^[LR]-"), "oblicua_frontal")
                                .otherwise("desconocida"))
      .withColumn("image_id", F.sha2(F.col("content"), 256))
      .select("image_id", "path", "filename", "bcs", "cow_id", "tipo_toma", "length", "modificationTime"))
# A partir de aquí ya no cargamos el contenido binario, solo los metadatos
df = df.cache()

df.groupBy("bcs", "tipo_toma").agg(F.count("*").alias("imagenes"),
                                   F.countDistinct("cow_id").alias("vacas")).orderBy("bcs", "tipo_toma").show()

+----+---------------+--------+-----+
| bcs|      tipo_toma|imagenes|vacas|
+----+---------------+--------+-----+
|3.25|cenital_trasera|      59|    8|
|3.25|oblicua_frontal|       1|    0|
| 3.5|cenital_trasera|      53|    6|
| 3.5|oblicua_frontal|       1|    0|
+----+---------------+--------+-----+



## Paso 3: Limpieza

A cada imagen le asignamos un motivo de exclusión, o nulo si es válida. Así queda registrado qué quitamos y por qué, en vez de borrar registros sin dejar rastro. Las reglas son:

1. Archivo vacío.
2. BCS fuera de las clases elegidas o que no se puede leer.
3. Sin ID de vaca (tomas L- y R-).
4. Vaca con más de un BCS distinto (etiqueta inconsistente).
5. Duplicado exacto (mismo hash): nos quedamos con la primera copia.

In [8]:
# Regla 4: vacas con más de un BCS
inconsistentes = (df.where(F.col("cow_id").isNotNull())
                    .groupBy("cow_id").agg(F.countDistinct("bcs").alias("n_bcs"))
                    .where("n_bcs > 1").select("cow_id", F.lit(True).alias("bcs_inconsistente")))

# Regla 5: duplicados por hash
w_dup = Window.partitionBy("image_id").orderBy("path")

clasif = (df.join(inconsistentes, "cow_id", "left")
            .withColumn("copia", F.row_number().over(w_dup))
            .withColumn("motivo_exclusion",
                F.when(F.col("length") == 0, "archivo_vacio")
                 .when(~F.col("bcs").isin(CLASES_BCS) | F.col("bcs").isNull(), "bcs_fuera_de_clases")
                 .when(F.col("cow_id").isNull(), "sin_id_vaca")
                 .when(F.col("bcs_inconsistente"), "bcs_inconsistente")
                 .when(F.col("copia") > 1, "duplicado")))

# Reporte de limpieza
clasif.groupBy(F.coalesce("motivo_exclusion", F.lit("OK")).alias("estado")) \
      .count().orderBy(F.desc("count")).show()

limpio = clasif.where(F.col("motivo_exclusion").isNull()).drop("copia", "bcs_inconsistente")
print(f"Imágenes válidas: {limpio.count()} | Vacas: {limpio.select('cow_id').distinct().count()}")

+-----------+-----+
|     estado|count|
+-----------+-----+
|         OK|  112|
|sin_id_vaca|    2|
+-----------+-----+

Imágenes válidas: 112 | Vacas: 14


## Paso 4: Muestreo por vaca

La muestra se toma por vaca y no por imagen, para no partir a un mismo animal. Dentro de cada clase de BCS elegimos VACAS_POR_CLASE vacas al azar con una ventana de Spark, y con SEED el resultado siempre es el mismo.

In [9]:
vacas = limpio.groupBy("cow_id").agg(F.first("bcs").alias("bcs"), F.count("*").alias("n_imagenes"))

if VACAS_POR_CLASE:
    w_cls = Window.partitionBy("bcs").orderBy(F.xxhash64("cow_id", F.lit(SEED)))
    vacas = vacas.withColumn("r", F.row_number().over(w_cls)).where(F.col("r") <= VACAS_POR_CLASE).drop("r")

assert vacas.count() > 0, ("No quedaron vacas válidas. Revisen el reporte del Paso 3: si casi todo es "
                          "bcs_inconsistente, el mismo número de vaca se repite en varias carpetas de BCS.")
muestra = limpio.join(vacas.select("cow_id"), "cow_id", "inner")

vacas.groupBy("bcs").agg(F.count("*").alias("vacas"), F.sum("n_imagenes").alias("imagenes"),
                         F.round(F.avg("n_imagenes"), 2).alias("img_por_vaca")).orderBy("bcs").show()

+----+-----+--------+------------+
| bcs|vacas|imagenes|img_por_vaca|
+----+-----+--------+------------+
|3.25|    8|      59|        7.38|
| 3.5|    6|      53|        8.83|
+----+-----+--------+------------+



## Paso 5: Partición k-fold agrupada por vaca

Por qué no usamos randomSplit: en el ejercicio de churn usamos randomSplit, que separa por fila. Aquí eso metería fotos de la misma vaca en entrenamiento y en prueba, y el modelo aprendería a reconocer a la vaca en vez de su condición corporal. Eso es fuga de datos.

Por eso usamos StratifiedGroupKFold (el grupo es la vaca y se estratifica por BCS) sobre la tabla de vacas, que tiene una fila por animal y sigue siendo chica aunque crezcan los datos. El resultado regresa a Spark como un mapa (create_map), así la unión se hace en el JVM sin crear DataFrames desde Python en los executors.

In [10]:
from sklearn.model_selection import StratifiedGroupKFold

vacas_pd = vacas.orderBy("cow_id").toPandas()
clase = (vacas_pd["bcs"] * 100).astype(int)          # 3.25 -> 325 (etiqueta discreta para estratificar)
k_eff = min(K, len(vacas_pd))

sgkf = StratifiedGroupKFold(n_splits=k_eff, shuffle=True, random_state=SEED)
vacas_pd["fold"] = -1
for f, (_, idx_prueba) in enumerate(sgkf.split(vacas_pd, clase, groups=vacas_pd["cow_id"])):
    vacas_pd.loc[idx_prueba, "fold"] = f

mapa_fold = F.create_map(*[F.lit(x) for par in zip(vacas_pd["cow_id"], vacas_pd["fold"].astype(int)) for x in par])
con_fold = muestra.withColumn("fold", mapa_fold[F.col("cow_id")])

# Revisamos que no haya fuga: ninguna vaca en más de un fold y ninguna imagen sin fold
fuga = con_fold.groupBy("cow_id").agg(F.countDistinct("fold").alias("n")).where("n > 1").count()
sin_fold = con_fold.where(F.col("fold").isNull()).count()
assert fuga == 0 and sin_fold == 0, f"Fuga: {fuga} vacas repartidas, {sin_fold} imágenes sin fold"
print(f"OK: {k_eff} folds, ninguna vaca repartida entre folds")

resumen_folds = (con_fold.groupBy("fold")
                 .pivot("bcs", CLASES_BCS).agg(F.countDistinct("cow_id"))
                 .join(con_fold.groupBy("fold").agg(F.countDistinct("cow_id").alias("vacas"),
                                                     F.count("*").alias("imagenes")), "fold")
                 .orderBy("fold"))
resumen_folds.show()

OK: 5 folds, ninguna vaca repartida entre folds
+----+----+----+-----+--------+
|fold|3.25| 3.5|vacas|imagenes|
+----+----+----+-----+--------+
|   0|   1|   2|    3|      29|
|   1|   3|NULL|    3|      21|
|   2|   2|   1|    3|      18|
|   3|   1|   2|    3|      29|
|   4|   1|   1|    2|      15|
+----+----+----+-----+--------+



Para entrenar, en cada fold f:
```python
train = con_fold.where(F.col("fold") != f)
prueba = con_fold.where(F.col("fold") == f)
```
Las imágenes que el Paso 6 marque como corruptas conservan su fold en la tabla, así que al entrenar con processed/imagenes.parquet hay que filtrar estado_imagen == "ok".

En el proyecto final, el dataset externo va solo en entrenamiento y la prueba lleva solo vacas del CAETEC.

## Paso 6: Transformación de imágenes

Para cada imagen de la muestra corregimos la orientación con el EXIF, la pasamos a RGB, la redimensionamos y la guardamos como JPEG sin metadatos. En las fotos del CAETEC esto quita las coordenadas GPS. Cada archivo se nombra con su hash para que el nombre no revele nada del rancho.

Esto corre en el driver con Pillow porque son pocas imágenes, y así evitamos UDFs de Python en el clúster. Con muchos más datos, este paso se haría con mapInPandas en los executors, leyendo y escribiendo en almacenamiento compartido (S3 o HDFS).

In [11]:
from PIL import Image, ImageOps
try:
    from pillow_heif import register_heif_opener; register_heif_opener()   # fotos HEIC del CAETEC
except ImportError:
    pass

clean_dir = Path(OUT_PATH, "clean", "images")
shutil.rmtree(clean_dir, ignore_errors=True)   # quita imágenes de una corrida anterior
filas = con_fold.select("image_id", "path", "bcs", "cow_id", "fold").toPandas()

def procesar(fila):
    src = fila.path.replace("file:", "", 1)
    dst = clean_dir / f"bcs={fila.bcs}" / f"{fila.image_id[:16]}.jpg"
    try:
        with Image.open(src) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            im.thumbnail((LADO_MAX, LADO_MAX))
            dst.parent.mkdir(parents=True, exist_ok=True)
            im.save(dst, "JPEG", quality=92)          # sin pasar exif, se guarda sin metadatos
            return pd.Series({"ruta_limpia": str(dst), "ancho": im.width, "alto": im.height, "estado_imagen": "ok"})
    except Exception as e:
        return pd.Series({"ruta_limpia": None, "ancho": None, "alto": None, "estado_imagen": f"corrupta: {e}"})

filas = pd.concat([filas, filas.apply(procesar, axis=1)], axis=1)
print(filas["estado_imagen"].value_counts())
print(filas[["ancho", "alto"]].describe().loc[["min", "max"]])

estado_imagen
ok    112
Name: count, dtype: int64
     ancho   alto
min  640.0  360.0
max  640.0  360.0


## Paso 7: Carga y verificación

Guardamos las tablas en Parquet, y también en CSV para revisarlas rápido, dentro de processed/. Como el clúster no tiene almacenamiento compartido, las escribimos desde el driver. En el ejercicio de churn, guardar desde los executors dejaba las partes repartidas entre las máquinas.

Para verificar, volvemos a leer las tablas y las comparamos con lo que procesó Spark. También recalculamos con hashlib el SHA-256 de algunos archivos originales para confirmar que coincide con el hash de Spark.

In [12]:
proc = Path(OUT_PATH, "processed"); proc.mkdir(parents=True, exist_ok=True)

imagenes_pd = clasif.drop("copia", "bcs_inconsistente").toPandas() \
    .merge(filas[["path", "fold", "ruta_limpia", "ancho", "alto", "estado_imagen"]], on="path", how="left")
imagenes_pd["path"] = imagenes_pd["path"].str.replace(r"^file:", "", regex=True)
imagenes_pd["modificationTime"] = imagenes_pd["modificationTime"].astype(str)
vacas_pd["fold"] = vacas_pd["fold"].astype(int)
resumen_pd = resumen_folds.toPandas()

for nombre, tabla in {"imagenes": imagenes_pd, "vacas": vacas_pd, "folds_resumen": resumen_pd}.items():
    tabla.to_parquet(proc / f"{nombre}.parquet", index=False)
    tabla.to_csv(proc / f"{nombre}.csv", index=False)

# 1) Conteos: lo que volvemos a leer tiene que coincidir con lo que calculó Spark
releido = pd.read_parquet(proc / "imagenes.parquet")
n_rel, n_ok_rel = len(releido), int(releido["fold"].notna().sum())
assert n_rel == n_raw, (n_rel, n_raw)
assert n_ok_rel == len(filas), (n_ok_rel, len(filas))
assert sum(1 for _ in clean_dir.rglob("*.jpg")) == int((filas["estado_imagen"] == "ok").sum())

# 2) Hashes: comparamos hashlib (Python) con sha2 (Spark) en 10 archivos al azar
chequeo = imagenes_pd.sample(min(10, len(imagenes_pd)), random_state=SEED)
for _, r in chequeo.iterrows():
    assert hashlib.sha256(Path(r.path).read_bytes()).hexdigest() == r.image_id, r.path

print(f"Carga verificada: {n_rel} registros, {n_ok_rel} imágenes con fold, hashes correctos")
print(sorted(p.name for p in proc.iterdir()))

Carga verificada: 114 registros, 112 imágenes con fold, hashes correctos
['folds_resumen.csv', 'folds_resumen.parquet', 'imagenes.csv', 'imagenes.parquet', 'vacas.csv', 'vacas.parquet']


## Paso 8: Muestra para el repositorio

In [13]:
sample_dir = Path(OUT_PATH, "sample");
if sample_dir.exists(): shutil.rmtree(sample_dir)
(sample_dir / "images").mkdir(parents=True)

ok = filas[filas["estado_imagen"] == "ok"]
sel = ok.groupby("fold").head(max(1, N_MUESTRA_REPO // k_eff))     # algunas imágenes de cada fold
for _, r in sel.iterrows():
    shutil.copy(r.ruta_limpia, sample_dir / "images" / Path(r.ruta_limpia).name)
sel.assign(archivo=sel["ruta_limpia"].map(lambda p: Path(p).name))[
    ["image_id", "archivo", "cow_id", "bcs", "fold", "ancho", "alto"]].to_csv(sample_dir / "muestra.csv", index=False)
shutil.copy(proc / "folds_resumen.csv", sample_dir)
print(f"{len(sel)} imágenes + muestra.csv + folds_resumen.csv en {sample_dir}/ -> copiar a data/sample/ del repo")

20 imágenes + muestra.csv + folds_resumen.csv en /content/drive/MyDrive/lecheros/salidas_etl/sample/ -> copiar a data/sample/ del repo


## Paso 9: Escalabilidad
| Ahora (demostración) | Con más datos |
|---|---|
| Archivos copiados a la misma ruta en cada máquina | Almacenamiento compartido: Amazon S3 (s3a://...) o HDFS |
| Imágenes procesadas con Pillow en el driver | mapInPandas en los executors, escribiendo directo a S3 |
| Tablas escritas desde el driver | df.write.parquet("s3a://.../processed/") desde Spark |
| Clúster de 3 máquinas (standalone) | Amazon EMR con autoescalado |

El código de los Pasos 1 a 5 no cambia, solo cambian MASTER_URL y DATA_PATH. La tabla de vacas sigue siendo chica (una fila por animal), así que asignar los folds en el driver no es problema.


## Paso 10: Cómo se escalaría al entrenamiento en clúster (propuesta, no se ejecuta)

Hoy el ETL termina en tablas Parquet y fotos limpias. A gran escala, el siguiente paso sería entrenar el modelo de regresión de BCS en el mismo clúster, leyendo esas salidas desde almacenamiento compartido:

1. **Datos:** las fotos crudas llegan a S3. El ETL (Pasos 1 a 9, con `mapInPandas` en el Paso 6) deja las fotos limpias y las tablas en `s3a://.../processed/`.
2. **Entrenamiento:** un clúster con GPUs (por ejemplo, Amazon EMR) entrena con dos niveles de paralelismo:
   - **Folds en paralelo:** los K folds son independientes, así que se entrenan al mismo tiempo, uno por executor. Con pocas vacas, como en nuestro caso, esta es la opción que más conviene.
   - **Un fold en varias GPUs:** si cada fold tuviera cientos de miles de fotos, `TorchDistributor` (Spark 3.4+) reparte un mismo entrenamiento de PyTorch entre varias GPUs con *DistributedDataParallel*.
3. **Predicción masiva:** con el modelo final, `predict_batch_udf` aplica el modelo a miles de fotos nuevas en paralelo y guarda el BCS predicho en Parquet.

En todos los casos la regla anti-fuga se mantiene: el fold viene de la tabla de vacas, así que una vaca nunca queda en entrenamiento y prueba a la vez.

```python
# ---------- Pseudocódigo: no se ejecuta en esta demostración ----------
TABLAS = "s3a://bcs-proyecto/processed/"

# Opción A: entrenar los K folds en paralelo, uno por executor
def entrenar_fold(f):
    tabla = leer_parquet(TABLAS + "imagenes.parquet")          # pandas/pyarrow desde S3
    tabla = tabla[tabla.estado_imagen == "ok"]
    train, val = tabla[tabla.fold != f], tabla[tabla.fold == f]
    modelo = crear_modelo()                                     # p. ej. ResNet con una salida de regresión (BCS)
    entrenar(modelo, train)
    mae = evaluar(modelo, val)
    guardar(modelo, f"s3://bcs-proyecto/modelos/fold_{f}.pt")
    return f, mae

resultados = spark.sparkContext.parallelize(range(K), K).map(entrenar_fold).collect()
print("MAE promedio de la validación cruzada:", sum(m for _, m in resultados) / K)

# Opción B: un fold muy grande repartido en varias GPUs
from pyspark.ml.torch.distributor import TorchDistributor

def entrenar_ddp(f):
    import torch.distributed as dist
    dist.init_process_group("nccl")    # TorchDistributor ya define RANK, WORLD_SIZE y MASTER_ADDR
    # cada proceso lee su parte de las fotos (DistributedSampler) y el modelo va en
    # DistributedDataParallel, que sincroniza los gradientes entre GPUs en cada paso
    ...
    if dist.get_rank() == 0:
        guardar_modelo_y_metricas()

TorchDistributor(num_processes=4, local_mode=False, use_gpu=True).run(entrenar_ddp, 0)

# Predicción masiva sobre fotos nuevas
from pyspark.ml.functions import predict_batch_udf
from pyspark.sql.types import FloatType

predecir_bcs = predict_batch_udf(cargar_modelo_y_predecir, return_type=FloatType(), batch_size=64)
nuevas = spark.read.parquet("s3a://bcs-proyecto/nuevas/")
nuevas.withColumn("bcs_predicho", predecir_bcs("pixeles")).write.parquet("s3a://bcs-proyecto/predicciones/")
```

| Pieza | Esta demostración | A gran escala |
|---|---|---|
| Datos | ~100 fotos en Google Drive | Miles de fotos en S3 |
| Cómputo | Colab con `local[*]` | EMR con nodos GPU y autoescalado |
| Validación cruzada | Folds asignados, sin entrenar | K entrenamientos en paralelo |
| Entrenamiento grande | No aplica | `TorchDistributor` (PyTorch DDP) |
| Predicción | No aplica | `predict_batch_udf` sobre Parquet |


## Interpretación
- Limpieza: [resumir la tabla del Paso 3: cuántas imágenes quitamos y por qué]
- Muestra: [N] vacas y [M] imágenes, con [x] imágenes por vaca en promedio
- Folds: [comentar cómo quedaron repartidas las vacas y las clases en cada fold del Paso 5]
- Tiempo local vs. clúster: [agregar si lo medimos]

In [14]:
spark.stop()